In [ ]:
!pip install streamlit supabase folium streamlit-folium pandas

In [ ]:
import streamlit as st
from supabase import create_client, Client
import folium
from streamlit_folium import st_folium
import pandas as pd

# 1. Configuración de la página para que se adapte a pantallas de celulares
st.set_page_config(page_title="Calendario de Montaña", page_icon="🏔️", layout="centered")

# 2. Conexión a Supabase (Reemplaza con tus credenciales del panel de Supabase)
SUPABASE_URL = "https://nmabblcjutzyzipbophi.supabase.co"
SUPABASE_KEY = "sb_publishable_c1eTQYTn6IXBGVkghPSXrA_JJwaMyJT"
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# Estilos CSS personalizados para que se vea genial en teléfonos
st.markdown("""
    <style>
    .card {
        background-color: #f8f9fa;
        padding: 15px;
        border-radius: 10px;
        border-left: 5px solid #2e7d32;
        margin-bottom: 15px;
    }
    .btn-whatsapp {
        background-color: #25D366;
        color: white !important;
        padding: 8px 12px;
        border-radius: 5px;
        text-decoration: none;
        font-weight: bold;
        display: inline-block;
        margin-top: 10px;
    }
    </style>
""", unsafe_allow_html=True)

st.title("🏔️ Próximas Salidas y Eventos")
st.subheader("Calendario de Actividades al Aire Libre")

# 3. Consultar los datos de eventos organizados por fecha
response = supabase.table("eventos_montana").select("*").order("fecha", desc=False).execute()
eventos = response.data

if not eventos:
    st.info("Por el momento no hay eventos programados. ¡Vuelve pronto!")
else:
    # 4. Mostrar los eventos en formato lista/tarjeta (Ideal para scroll en celular)
    for ev in eventos:
        with st.container():
            # Crear la tarjeta informativa
            st.markdown(f"""
            <div class="card">
                <h3>{ev['titulo']}</h3>
                <p><b>📅 Fecha:</b> {ev['fecha']} | <b>👤 Organiza:</b> {ev['organizador']}</p>
                <p>{ev['descripcion']}</p>
            </div>
            """, unsafe_allow_html=True)
            
            # Columnas interactivas: Botón de WhatsApp y Mapa de ubicación
            col1, col2 = st.columns([1, 1])
            
            with col1:
                # Generar el link directo a la API de WhatsApp con un mensaje predeterminado
                mensaje_wa = f"Hola, me interesa el evento de {ev['titulo']} del día {ev['fecha']}"
                link_wa = f"https://wa.me/{ev['telefono_whatsapp']}?text={uri_encode(mensaje_wa)}" # Necesitas importar urllib.parse si usas caracteres especiales, o texto simple:
                link_wa = f"https://wa.me/{ev['telefono_whatsapp']}?text=Hola,%20me%20interesa%20el%20evento%20de%20{ev['titulo']}"
                
                st.markdown(f'<a href="{link_wa}" target="_blank" class="btn-whatsapp">💬 Contactar por WhatsApp</a>', unsafe_allow_html=True)
            
            with col2:
                # Si el evento tiene coordenadas, pintar un mapa interactivo compacto
                if ev['latitud'] and ev['longitud']:
                    with st.expander("📍 Ver mapa de ubicación"):
                        m = folium.Map(location=[ev['latitud'], ev['longitud']], zoom_start=12, control_scale=True)
                        folium.Marker(
                            [ev['latitud'], ev['longitud']], 
                            popup=ev['titulo'], 
                            tooltip="Punto de encuentro / Cumbre"
                        ).add_to(m)
                        st_folium(m, height=200, width=300)
            
            st.markdown("---")